# Lab — OPC UA Machinery: Read a Machine's Identity
**Certificate 03 · Automatic Identification Technology**  |  [← Back to the Lab Hub](../../index.ipynb)

Auto-identification is not only barcodes and RFID. Under **OPC UA for Machinery**, a machine carries a standard **Identification** profile (an electronic nameplate) and *identifies itself* over the network — make, model, serial number, revisions, capabilities. This lab reads that nameplate.

> **Skeleton:** this notebook boots a small self-contained OPC UA server so it runs anywhere (incl. Colab). Swap in your own server endpoint where marked to read a real machine.

## Objectives
By the end of this lab you will be able to:
- Explain what the OPC UA for Machinery **Identification** profile is and why it is "auto-ID for machines".
- Connect an OPC UA client to a server and browse to an Identification object.
- Read the standard nameplate properties and print a machine "identity card".

## Background — the Identification nameplate
The Machinery companion spec (OPC 40001-1) reuses the OPC UA **DI** *Identification* FunctionalGroup. Standard properties include:

| Property | Meaning |
|---|---|
| `Manufacturer` | Who built it |
| `Model` | Product/model name |
| `ProductCode` | Orderable product code |
| `SerialNumber` | Unique unit serial |
| `HardwareRevision` / `SoftwareRevision` / `DeviceRevision` | Revisions |
| `ProductInstanceUri` | Globally-unique id for *this* unit |
| `DeviceClass` | Kind of machine |
| `YearOfConstruction` | Build year |

_TODO (next run): map these against the exact BrowseNames in your `Opc.Ua.Machinery.NodeSet2.xml`._

## Setup
Install the OPC UA library (`asyncua`). Colab needs `nest_asyncio` so we can run an async server + client inside the notebook loop.

In [ ]:
!pip -q install asyncua nest_asyncio

In [ ]:
import asyncio, nest_asyncio
from asyncua import Server, Client, ua
nest_asyncio.apply()
print("asyncua ready")

## 1 · Demo server (self-contained)
Boots an OPC UA server exposing one machine (`PressBrake_01`) with an **Identification** object + a few live process values — mirroring your `opc_Server_machineryType.py`.

_Replace this whole section with your real server later; the client in step 2 only needs the endpoint URL._

In [ ]:
ENDPOINT = "opc.tcp://127.0.0.1:4840/intellexfabrica/cert03/"

NAMEPLATE = {
    "Manufacturer":       "Acme Machine Tools",
    "Model":              "PB-250 Press Brake",
    "ProductCode":        "PB250-STD",
    "SerialNumber":       "SN-000173",
    "HardwareRevision":   "3.1",
    "SoftwareRevision":   "12.4.0",
    "DeviceRevision":     "A",
    "ProductInstanceUri": "urn:acme:pressbrake:SN-000173",
    "DeviceClass":        "PressBrake",
    "YearOfConstruction": 2022,
}

async def start_demo_server():
    server = Server()
    await server.init()
    server.set_endpoint(ENDPOINT)
    idx = await server.register_namespace("http://intellexfabrica.com/cert03-machinery")
    machine = await server.nodes.objects.add_object(idx, "PressBrake_01")
    ident = await machine.add_object(idx, "Identification")
    for name, val in NAMEPLATE.items():
        await ident.add_property(idx, name, val)
    # a few live process values, like the machinery server in Resources/05_OPC
    await machine.add_variable(idx, "Temperature", 25.0)
    await machine.add_variable(idx, "Pressure", 100.0)
    await machine.add_variable(idx, "Speed", 1500.0)
    await server.start()
    return server

server = await start_demo_server()
print("Demo OPC UA server running at", ENDPOINT)

## 2 · Read the machine's identity
Point the client at a server, browse for an `Identification` object under any machine, and read every nameplate property.

**To read a real machine:** set `SERVER_URL` to your endpoint (e.g. `opc.tcp://172.20.1.12:4840`) — the server must be reachable from wherever this notebook runs.

In [ ]:
SERVER_URL = ENDPOINT   # <-- change to your own machinery server endpoint

async def read_identity(url):
    async with Client(url=url) as client:
        for machine in await client.nodes.objects.get_children():
            mname = (await machine.read_browse_name()).Name
            for child in await machine.get_children():
                if (await child.read_browse_name()).Name == "Identification":
                    print(f"\n=== Identity card: {mname} ===")
                    for prop in await child.get_children():
                        pn  = (await prop.read_browse_name()).Name
                        val = await prop.read_value()
                        print(f"  {pn:20} {val}")

await read_identity(SERVER_URL)

## 3 · Stop the demo server
Run this when done (or before re-running step 1).

In [ ]:
await server.stop()
print("server stopped")

## Debrief
- Why is a self-describing **Identification** profile a form of automatic identification — and what does it give you that a barcode on the frame does not?
- `ProductInstanceUri` vs `SerialNumber` — why have both?
- If every machine on a line exposed this profile, what could a plant system do automatically on start-up?

_Next run: extend with the real NodeSet, add `MachineryItemState`, and point the client at a live server._